# HW2 檢索增強生成（RAG）與評量

> 說明：<https://github.com/chang-ye-tu/genai/blob/main/hw/hw2.md>　文件庫：`hw/hw2/corpus.md`（18 段，已內嵌於本筆記本，請勿修改）
> 核心段落：第 1–5 節（實作測驗只出這些段落的題目）；第 6 節為選做，不計分。
> 執行階段請選 T4 GPU；第 1–3 節不需要 GPU，第 4–5 節需要。
> 本次作業另有**一頁「檢索失敗案例分析」**要上傳到 iLearn 同儕互評活動（見說明第 6 節）。


In [ ]:
# @title 第 0 節：安裝與載入文件庫（文件庫已內嵌，與 hw/hw2/corpus.md 相同）
%pip -q install transformers==5.16.1 sentence-transformers==6.0.1 accelerate==1.14.0
import re, numpy as np, torch
from sentence_transformers import SentenceTransformer
import platform, importlib.metadata as _meta
def _v(p):
    try: return _meta.version(p)
    except Exception: return "missing"  # metadata 查不到時印 missing；若同一格更早的 import 已失敗，程式到不了這裡，check_submissions 會判「無版本資訊／執行錯誤」
print("VERSIONS", "python=" + platform.python_version(), "torch=" + torch.__version__, *[p + "=" + _v(p) for p in ["transformers", "sentence-transformers", "accelerate"]])
CORPUS = '# HW2 知識庫：本課程的規則與教材摘要（凍結版，2026-09）\n\n> 這是 HW2「檢索增強生成」的固定文件庫。每個段落以 `[P01]`、`[P02]`…編號，筆記本會以段落為單位切塊（chunk）、建立向量索引並檢索。請勿修改內容，否則實作測驗的答案會對不上。\n\n[P01] 「生成式人工智慧：理論與實務」是逢甲大學 115 學年度第 1 學期開設的 2 學分選修課，採全程非同步線上教學，教學平台為 iLearn。每個單元於星期一發布，共 13 個單元，課程期間為 2026 年 9 月 8 日至 12 月 28 日；9 月 8 日至 13 日為導覽週，13 個單元自 9 月 14 日起發布。\n\n[P02] 本課程的主要教材是臺灣大學李宏毅教授公開的中文課程影片，包括《生成式人工智慧與機器學習導論》2025 秋、《生成式 AI 導論》2024 春，以及《機器學習》2025 春與 2026 春的部分單元；輔助教材為 3Blue1Brown 的深度學習系列影片。所有影片與投影片皆以原始網址連結引用並註明出處。\n\n[P03] 單元學習流程：閱讀學習指引與教材，完成本單元測驗（發布當天即可作答）。第 1 至 12 單元的活動都在發布後第二個星期日 23:59 截止；第 13 單元例外，於 12 月 28 日星期一截止。只有第 1、2、3、5、6、9、10、11、13 單元有討論：發表 1 篇主題、在其他同學的主題下回覆 1 則，完成後「討論簽到」小測驗才會開放，在簽到題選「是」並交卷即得 1 分。\n\n[P04] 單元測驗由題庫隨機抽出 10 題選擇題，作答 1 次；答錯可以查看提示後再試，最多 3 次，每次扣該題三分之一的分數；測驗關閉後系統會顯示正確答案與每個選項的解說。\n\n[P05] 成績比重：單元測驗 45%、討論參與 5%、實作作業 45%、作業互評 5%。沒有期中考、期末考，也沒有專題。\n\n[P06] 單元測驗共 12 次（第 1 單元至第 12 單元各一次，第 13 單元沒有測驗），成績取最佳 10 次平均；討論簽到共 9 次（第 1、2、3、5、6、9、10、11、13 單元），取最佳 7 次平均。未作答以 0 分計入後，單元測驗與討論簽到各捨去最低 2 次。測驗、討論與簽到逾期不受理。\n\n[P07] 實作作業共七次：HW1 語言模型初體驗（9 月 21 日發布、10 月 11 日截止）、HW2 檢索增強生成與評量（10 月 5 日發布、10 月 25 日截止）、HW3 從零實作注意力與 GPT 模型（10 月 19 日發布、11 月 8 日截止）、HW4 評量推理模型（11 月 2 日發布、11 月 22 日截止）、HW5 預訓練小型 GPT（11 月 16 日發布、12 月 6 日截止）、HW6 微調與 LoRA（11 月 30 日發布、12 月 20 日截止）、HW7 安全性攻防實驗（12 月 7 日發布、12 月 27 日截止）。\n\n[P08] 每次實作作業以 Colab 筆記本繳交加上實作測驗計分，繳交筆記本後測驗才會開放。實作測驗每次 9 至 11 題，作答 1 次，答錯可以查看提示後再試；七次成績取最佳 5 次平均。HW1 至 HW6 截止後有 7 天寬限期，寬限期內繳交不扣分，寬限期後不受理；HW7 因為 12 月 29 日是課程考核截止日，寬限只到 12 月 28 日（1 天）。\n\n[P09] 作業互評只有 HW2 有：把檢索失敗案例分析寫成一頁 PDF，於 10 月 25 日前上傳同儕互評活動，10 月 26 日至 11 月 1 日每人匿名評閱 3 位同學的作品。系統自動計算繳交評分（占 80%）與評量評分（占 20%）；完全未評閱者評量評分為 0，未完成 3 份者由教師依評分報告手動扣除。互評活動沒有寬限期。\n\n[P10] 第 13 單元（12 月 21 日發布）沒有新增必看教材，只有課程總結（學習指引頁面）、學習成果分享討論區與期末回饋單，另有兩支選看的延伸影片；沒有測驗。12 月 29 日為課程考核截止日，所有成績於當日結算。\n\n[P11] 本課程鼓勵使用生成式 AI 工具協助學習，但繳交物必須附上 AI 使用聲明，說明使用了什麼工具、做了什麼，以及自己做了什麼；未附聲明以未繳交論；作業互評的一頁 PDF 頁尾也要有一行 AI 使用聲明，缺了只由評閱者把 R1 評為 0 級，這是「以未繳交論」的唯一例外。所有作業與互評皆為個人作品，測驗須本人獨立作答。\n\n[P12] 互動管道包括：公告區（強制訂閱）、課程問題 Q&A 討論區（教師於 2 個工作天內回覆，同學也可互相解答）、有討論的單元的討論區、第 13 單元的學習成果分享討論區，以及 Email。本課程沒有教學助理，也沒有同步的線上時段。Email 主旨請註明「生成式AI」與學號。\n\n[P13] 語言模型的運作方式是文字接龍：把輸入當成未完成的句子，反覆預測下一個 token 的機率分佈並依機率抽樣，直到產生結束符號。同一個問題每次答案不同，是因為抽樣具有隨機性。\n\n[P14] 幻覺（hallucination）不是故障，而是文字接龍的自然結果：當輸入資訊不足時，模型仍會產生看似合理但不存在的內容。減少幻覺的方法包括提供相關文件（檢索增強生成）、要求引用來源、降低隨機性，以及允許模型回答不知道。\n\n[P15] 檢索增強生成（RAG）的流程是：先把文件切成段落並轉成向量存入索引；使用者提問時，把問題轉成向量，找出最相似的幾個段落；再把這些段落連同問題一起放進提示，讓語言模型根據段落作答。\n\n[P16] 評量生成式 AI 時常見的陷阱包括：測試資料曾出現在訓練資料中（資料污染）、評分標準不一致、只看單一指標、以及用語言模型當評審時的偏好偏差（例如偏好較長或位置在前的答案）。設計評測時應使用模型沒看過的題目，並同時採用多個指標。\n\n[P17] 安全性議題包括偏見、幻覺、越獄（jailbreak）與提示注入（prompt injection）。提示注入是指攻擊者把指令藏在模型會讀到的內容（例如網頁或文件）中，誘使模型執行非使用者本意的動作。安全性實作僅能在自己的帳號、對自己準備的資料進行測試。\n\n[P18] 期中課程回饋單於 11 月 2 日至 11 月 15 日開放，期末課程回饋單於 12 月 21 日至 12 月 28 日開放，皆為匿名填答（教師介面與報表不顯示身分，平台資料庫仍有紀錄），內容涵蓋課程內容、教學活動與平台服務三個面向，作為課程持續改善的依據。\n'
paras = re.findall(r"^\[(P\d\d)\]\s*(.+?)(?=\n\s*\n|\Z)", CORPUS, flags=re.S | re.M)
ids = [p[0] for p in paras]; docs = [re.sub(r"\s+", " ", p[1]).strip() for p in paras]
print("段落數：", len(docs))
for i, d in zip(ids, docs):
    print(i, len(d), "字 |", d[:40], "…")


## 第 2 節 向量化與相似度

e5 模型的約定：問題前加 `query: `，段落前加 `passage: `。向量正規化後，cosine 相似度就是內積。


In [ ]:
emb = SentenceTransformer("intfloat/multilingual-e5-small", revision="614241f622f53c4eeff9890bdc4f31cfecc418b3")
D = emb.encode(["passage: " + d for d in docs], normalize_embeddings=True)
print("embeddings.shape =", D.shape, "→ 每個向量", D.shape[1], "維")
PP = D @ D.T
np.fill_diagonal(PP, 0)
a, b = np.unravel_index(PP.argmax(), PP.shape)
print(f"most similar pair: {ids[a]} 與 {ids[b]}，相似度 {PP[a, b]:.3f}")
print("P13-P14:", round(float(D[12] @ D[13]), 3), "| P01-P17:", round(float(D[0] @ D[16]), 3), "| P03-P04:", round(float(D[2] @ D[3]), 3))


In [ ]:
# 試試看：拿掉前綴，相似度會變嗎？
q = "什麼是提示注入？"
with_prefix = emb.encode(["query: " + q], normalize_embeddings=True) @ D.T
D_np = emb.encode(docs, normalize_embeddings=True)
no_prefix = emb.encode([q], normalize_embeddings=True) @ D_np.T
print("有前綴 top-3：", [(ids[j], round(float(with_prefix[0, j]), 3)) for j in np.argsort(-with_prefix[0])[:3]])
print("無前綴 top-3：", [(ids[j], round(float(no_prefix[0, j]), 3)) for j in np.argsort(-no_prefix[0])[:3]])


✍️ **請回答 2-1**：最相似的那一對段落為什麼相似？它們談的是同一件事嗎？這對 RAG 有什麼影響？

（在這裡作答）


## 第 3 節 檢索：每個問題的前 3 名段落


In [ ]:
QUESTIONS = {
 "Q01": "單元測驗可以作答幾次？", "Q02": "實作作業的成績怎麼計算？", "Q03": "什麼是提示注入？",
 "Q04": "教師多久會回覆 Q&A 討論區的問題？", "Q05": "為什麼同一個問題每次答案不同？", "Q06": "HW2 的截止日期是哪一天？",
 "Q07": "評量生成式 AI 有哪些陷阱？", "Q08": "這門課有幾學分？", "Q09": "RAG 的流程是什麼？",
 "Q10": "逾期繳交實作作業會怎麼樣？", "Q11": "減少幻覺有哪些方法？", "Q12": "期中回饋單什麼時候開放？",
}
Q = emb.encode(["query: " + q for q in QUESTIONS.values()], normalize_embeddings=True)
S = Q @ D.T
def retrieve(i, k=3):
    order = np.argsort(-S[i])[:k]
    return [(ids[j], round(float(S[i, j]), 3)) for j in order]
for i, (k, q) in enumerate(QUESTIONS.items()):
    top = retrieve(i)
    print(f"{k} {q:<22} → {top}   (第1-第2 差距 {top[0][1]-top[1][1]:.3f})")


✍️ **請回答 3-1**：哪些問題的第 1 名不是真正含答案的段落（提示：看 Q05，含答案的是 P13）？這種失敗是「字面相似」還是「語意相似」造成的？你會怎麼改善？這一題的觀察就是你要寫成一頁「檢索失敗案例分析」的材料。

（在這裡作答）


## 第 4 節 生成：有檢索 vs 無檢索（需要 GPU）


In [ ]:
# @title 載入生成模型 Qwen2.5-1.5B-Instruct
%pip -q install transformers==5.16.1 accelerate==1.14.0
import torch, transformers
USE_SMALL = False  # @param {type:"boolean"}  Colab 額度不足時改 True（實作測驗的數值題請以 1.5B 為準）
MODEL, REV = ("Qwen/Qwen2.5-0.5B-Instruct", "7ae557604adf67be50417f59c2c2f167def9a775") if USE_SMALL else ("Qwen/Qwen2.5-1.5B-Instruct", "989aa7980e4cf806f80c7fef2b1adb7bc71aa306")
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL, revision=REV)
DTYPE = torch.float32 if "fp16" == "fp32" else (torch.float16 if device == "cuda" else torch.float32)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, dtype=DTYPE).to(device).eval()
print("模型：", MODEL, "| 裝置：", device, "| dtype：", DTYPE)
# 版本已在第 0 節印出（VERSIONS）
print("詞彙表大小（tokenizer）：", len(tok), "| eos token：", tok.eos_token, tok.eos_token_id, "| pad：", tok.pad_token)

def chat_prompt(user, system=None, history=None):
    """把訊息套上對話模板，回傳模型實際看到的字串。"""
    msgs = []
    if system is not None:
        msgs.append({"role": "system", "content": system})
    for u, a in (history or []):
        msgs += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": user})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(prompt, max_new_tokens=64, temperature=0.0, top_p=1.0, raw=False):
    """raw=True 表示 prompt 已是完整字串（不套模板）。temperature=0 代表 greedy。"""
    text = prompt if raw else chat_prompt(prompt)
    ids = tok(text, return_tensors="pt").to(device)
    kw = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    if temperature > 0:
        kw.update(do_sample=True, temperature=temperature, top_p=top_p)
    else:
        kw.update(do_sample=False)
    out = model.generate(**ids, **kw)
    return tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()


In [ ]:
QA = [
 ('單元測驗可以作答幾次？', '1 次（答錯可看提示再試）', ['1 次']),
 ('HW2 的截止日期是哪一天？', '10 月 25 日', ['10 月 25']),
 ('這門課有幾學分？', '2 學分', ['2 學分']),
 ('實作作業一共有幾次？', '七次（7 次）', ['七次']),
 ('期中回饋單什麼時候開放？', '11 月 2 日至 11 月 15 日', ['11 月 2', '11 月 15']),
 ('教師多久會回覆 Q&A 討論區的問題？', '2 個工作天內', ['2 個工作天']),
 ('HW2 的檢索失敗案例分析每人要互評幾份？', '3 份', ['3 份', '3 位']),
 ('單元測驗成績取最佳幾次？', '10 次', ['10 次']),
 ('逾期繳交實作作業會怎麼樣？', 'HW1 至 HW6 截止後有 7 天寬限期，寬限期內繳交不扣分，寬限期後不受理；HW7 寬限只到 12 月 28 日（1 天）', ['7 天', '寬限期']),
 ('討論簽到要先完成什麼？', '在討論區發表 1 篇主題並回覆 1 位同學', ['主題', '回覆']),
]
def rag_prompt(q, k=3):
    qv = emb.encode(["query: " + q], normalize_embeddings=True)[0]
    top = np.argsort(-(D @ qv))[:k]
    ctx = "\n".join(f"[{ids[j]}] {docs[j]}" for j in top)
    return (f"下面是幾段課程規則。請只依據這些段落，用一句完整的繁體中文回答問題，並在句尾用括號標註引用的段落編號，例如「……（P04）」；"
            f"若段落中沒有答案，請回答「段落中沒有提到」。\n\n{ctx}\n\n問題：{q}\n回答：")

rows = []
for q, ref, kws in QA:
    a_no = generate(q, max_new_tokens=60)
    a_rag = generate(rag_prompt(q), max_new_tokens=80)
    rows.append((q, ref, a_no, a_rag, kws))
    print("Q:", q, "\n  無檢索:", a_no, "\n  有檢索:", a_rag)


## 第 5 節 評量：關鍵字比對（啟發式）vs LLM 當評審

關鍵字比對很寬鬆：出現任一關鍵字就算對；LLM 評審能看語意但可能有偏差。兩者都不是「精確比對」。


In [ ]:
def kw_score(ans, kws):
    # 關鍵字比對（寬鬆的啟發式）：只要出現任一關鍵字就算對，會把「只提到日期但意思錯」的回答也判對——這正是第 5 節要你觀察的限制
    return int(any(k in ans for k in kws))
def judge(q, ref, ans):
    p = (f"你是評分助教。問題：{q}\n標準答案：{ref}\n學生回答：{ans}\n"
         "請判斷學生回答是否與標準答案一致，只輸出「正確」「部分正確」或「錯誤」三者之一。")
    out = generate(p, max_new_tokens=8)
    return "正確" if out.startswith("正確") else ("部分正確" if "部分" in out else "錯誤")
tot = {"無檢索-關鍵字": 0, "有檢索-關鍵字": 0, "無檢索-評審": 0, "有檢索-評審": 0}
print(f"{'問題':<22}{'無檢索kw':>8}{'有檢索kw':>8}  無檢索評審  有檢索評審")
for q, ref, a_no, a_rag, kws in rows:
    k1, k2 = kw_score(a_no, kws), kw_score(a_rag, kws)
    j1, j2 = judge(q, ref, a_no), judge(q, ref, a_rag)
    tot["無檢索-關鍵字"] += k1; tot["有檢索-關鍵字"] += k2
    tot["無檢索-評審"] += (j1 == "正確"); tot["有檢索-評審"] += (j2 == "正確")
    print(f"{q:<22}{k1:>8}{k2:>8}  {j1:<8}  {j2}")
print({k: f"{v}/{len(rows)}" for k, v in tot.items()})


✍️ **請回答 5-1**：無檢索時模型答錯的題目，錯得像什麼（幻覺？含糊？迴避？）；有檢索仍答錯的題目，是檢索錯還是生成錯？LLM 評審在哪一題和你的看法不同？關鍵字比對有沒有把錯的回答判成對（例如只提到日期）？

（在這裡作答）


## 第 6 節（選做）換成你自己的文件

把 `docs` 換成你自己的段落（例如某門課的講義），重跑第 2–5 節。


## ✍️ AI 使用聲明（必填）

| 項目 | 內容 |
|------|------|
| 使用的工具 | （例如：ChatGPT 免費版、Colab 內建 Gemini） |
| 用在哪些工作 | （例如：解釋錯誤訊息、幫我看懂某一格程式） |
| 我自己完成的部分 | （例如：全部執行、所有 ✍️ 回答） |
| 我如何驗證 AI 的說法 | （例如：實際執行、對照投影片） |

姓名／學號：
